In [3]:
from pathlib import Path
import numpy as np
import pandas as pd

# Student information
LAST_NAME = "Acapulco"
STUDENT_ID = "TUPM-26-1874"

id_digits = "".join(c for c in STUDENT_ID if c.isdigit())

SEED_DIGIT = int(id_digits[-1])
ID_SUM = sum(int(c) for c in id_digits)
NAME_LENGTH = len(LAST_NAME)
RANDOM_SEED = int(id_digits) % (2**32)

np.set_printoptions(precision=4, suppress=True)

# Locate the project folders
CURRENT_DIR = Path.cwd()
DATA_DIR = None
APPENDIX_DIR = None

for folder in [CURRENT_DIR, *CURRENT_DIR.parents]:

    if (folder / "data" / "secom" / "secom.data").is_file():
        DATA_DIR = folder / "data" / "secom"

    elif (folder / "secom" / "secom.data").is_file():
        DATA_DIR = folder / "secom"

    if (folder / "data" / "Lab7 Appendix").is_dir():
        APPENDIX_DIR = folder / "data" / "Lab7 Appendix"

    elif (folder / "Lab7 Appendix").is_dir():
        APPENDIX_DIR = folder / "Lab7 Appendix"

    if DATA_DIR is not None and APPENDIX_DIR is not None:
        break

if DATA_DIR is None:
    raise FileNotFoundError("Cannot find the secom dataset folder.")

if APPENDIX_DIR is None:
    raise FileNotFoundError("Cannot find the Lab7 Appendix folder.")

OUTPUT_DIR = APPENDIX_DIR / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Dataset paths
measurement_path = DATA_DIR / "secom.data"
label_path = DATA_DIR / "secom_labels.data"
description_path = DATA_DIR / "secom.names"

for path in [measurement_path, label_path, description_path]:
    if not path.is_file():
        raise FileNotFoundError(f"Missing dataset file: {path}")

# Load SECOM measurements and labels
raw_measurements = np.genfromtxt(measurement_path, dtype=float)
labels = np.loadtxt(label_path, usecols=0, dtype=int)

# Verify dataset correspondence
assert raw_measurements.ndim == 2
assert len(raw_measurements) == len(labels)
assert np.isin(labels, [-1, 1]).all()

# Identify usable features
missing = np.isnan(raw_measurements)
observed_count = (~missing).sum(axis=0)

minimum = np.min(
    np.where(missing, np.inf, raw_measurements),
    axis=0
)

maximum = np.max(
    np.where(missing, -np.inf, raw_measurements),
    axis=0
)

all_missing = observed_count == 0
constant = (observed_count > 0) & (minimum == maximum)

usable_indices = np.flatnonzero(~(all_missing | constant))

# Create feature names
feature_names = [
    f"Feature_{i:03d}" for i in range(raw_measurements.shape[1])
]

# Create labeled DataFrame
df = pd.DataFrame(raw_measurements.copy(), columns=feature_names)
df["Original_Record_ID"] = np.arange(len(df))
df["Result"] = labels
df["Outcome"] = np.where(labels == -1, "Pass", "Fail")
df["Is_Fail"] = labels == 1

print("STUDENT:", LAST_NAME)
print("STUDENT ID:", STUDENT_ID)
print("SEED DIGIT:", SEED_DIGIT)
print("ID SUM:", ID_SUM)
print("RANDOM SEED:", RANDOM_SEED)
print("Dataset Shape:", raw_measurements.shape)
print("Pass Records:", np.sum(labels == -1))
print("Fail Records:", np.sum(labels == 1))
print("Missing Values:", missing.sum())
print("Unusable Features:", all_missing.sum() + constant.sum())
print("Usable Features:", len(usable_indices))
print("Output Folder:", OUTPUT_DIR)
print("SETUP COMPLETE!")

STUDENT: Acapulco
STUDENT ID: TUPM-26-1874
SEED DIGIT: 4
ID SUM: 28
RANDOM SEED: 261874
Dataset Shape: (1567, 590)
Pass Records: 1463
Fail Records: 104
Missing Values: 41951
Unusable Features: 116
Usable Features: 474
Output Folder: c:\Users\GABRIEL LAWRENCE\Lab7 NumPy_Pandas_Acapulco\Lab7 Appendix\output
SETUP COMPLETE!


In [1]:

from pathlib import Path
import numpy as np
import pandas as pd

# STUDENT CONFIGURATION
LAST_NAME = "Acapulco"
STUDENT_ID = "TUPM-26-1874"

id_digits = "".join(d for d in STUDENT_ID if d.isdigit())

SEED_DIGIT = int(id_digits[-1])
ID_SUM = sum(map(int, id_digits))
NAME_LENGTH = len(LAST_NAME)

RANDOM_SEED = int(id_digits) % (2**32)

verification_value = ID_SUM * NAME_LENGTH + SEED_DIGIT

np.set_printoptions(precision=4, suppress=True)

print("Student:", LAST_NAME)
print("Student ID:", STUDENT_ID)
print("Seed Digit:", SEED_DIGIT)
print("ID Sum:", ID_SUM)
print("Name Length:", NAME_LENGTH)
print("Random Seed:", RANDOM_SEED)
print("Verification Value:", verification_value)


# DATASET PATHS
WORK_DIR = Path.cwd()

if (WORK_DIR / "secom").exists():
    DATA_DIR = WORK_DIR / "secom"
    APPENDIX_DIR = WORK_DIR / "Lab7 Appendix"

elif (WORK_DIR.parent / "secom").exists():
    DATA_DIR = WORK_DIR.parent / "secom"
    APPENDIX_DIR = WORK_DIR

else:
    raise FileNotFoundError("SECOM folder not found.")

OUTPUT_DIR = APPENDIX_DIR / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

measurement_path = DATA_DIR / "secom.data"
label_path = DATA_DIR / "secom_labels.data"
description_path = DATA_DIR / "secom.names"

for path in [measurement_path, label_path, description_path]:
    assert path.is_file(), f"Missing dataset: {path}"

# LOAD DATA
raw_measurements = np.genfromtxt(
    measurement_path,
    dtype=float
)

labels = np.loadtxt(
    label_path,
    usecols=0,
    dtype=int
)

assert raw_measurements.ndim == 2
assert len(labels) == len(raw_measurements)
assert np.isin(labels, [-1, 1]).all()

print("\nDataset Loaded Successfully")
print("Measurement Shape:", raw_measurements.shape)
print("Labels Shape:", labels.shape)
print("Output Folder:", OUTPUT_DIR.resolve())

Student: Acapulco
Student ID: TUPM-26-1874
Seed Digit: 4
ID Sum: 28
Name Length: 8
Random Seed: 261874
Verification Value: 228

Dataset Loaded Successfully
Measurement Shape: (1567, 590)
Labels Shape: (1567,)
Output Folder: C:\Users\GABRIEL LAWRENCE\Lab7 NumPy_Pandas_Acapulco\Lab7 Appendix\output


In [2]:

# MISSION 1
# VECTORIZED MEASUREMENT QUALITY ANALYSIS

# IDENTIFY MISSING AND CONSTANT FEATURES
missing = np.isnan(raw_measurements)

observed = (~missing).sum(axis=0)

all_missing = observed == 0

observed_min = np.where(
    missing, np.inf, raw_measurements
).min(axis=0)

observed_max = np.where(
    missing, -np.inf, raw_measurements
).max(axis=0)

constant = (
    (observed > 0) &
    (observed_min == observed_max)
)

usable_indices = np.flatnonzero(
    ~(all_missing | constant)
)

usable_data = raw_measurements[:, usable_indices].copy()

feature_names = [
    f"Feature_{i:03d}" for i in usable_indices
]

# REPRODUCIBLE FEATURE SELECTION
rng = np.random.default_rng(RANDOM_SEED)

selected_count = 5 + (SEED_DIGIT % 3)

selected_positions = rng.choice(
    len(usable_indices),
    size=selected_count,
    replace=False
)

selected_data = usable_data[:, selected_positions]

selected_features = [
    feature_names[i] for i in selected_positions
]

# NUMPY STATISTICS
feature_means = np.nanmean(selected_data, axis=0)
feature_medians = np.nanmedian(selected_data, axis=0)
feature_sds = np.nanstd(selected_data, axis=0, ddof=1)

missing_percentages = (
    np.isnan(selected_data).mean(axis=0) * 100
)

# ROBUST STANDARDIZATION USING MAD
medians = np.nanmedian(selected_data, axis=0)

mad = np.nanmedian(
    np.abs(selected_data - medians),
    axis=0
)

safe_mad = np.where(mad > 0, mad, 1.0)

robust_z = (
    0.6745 * (selected_data - medians)
    / safe_mad
)

# HANDLE ZERO MAD
robust_z[:, mad == 0] = 0

# IDENTIFY UNUSUAL MEASUREMENTS
flags = (
    np.abs(robust_z) > 3.5
) & ~np.isnan(selected_data)

flagged_counts = flags.sum(axis=0)

flagged_percentages = (
    flags.sum(axis=0) /
    np.maximum((~np.isnan(selected_data)).sum(axis=0), 1)
) * 100

# CREATE SUMMARY
quality_summary = pd.DataFrame({
    "Feature": selected_features,
    "Mean": feature_means,
    "Median": feature_medians,
    "Sample_SD": feature_sds,
    "Missing_Percentage": missing_percentages,
    "Flagged_Count": flagged_counts,
    "Flagged_Percentage": flagged_percentages
})

quality_summary = quality_summary.sort_values(
    by="Flagged_Percentage",
    ascending=False
)

# EXPORT CSV
quality_path = OUTPUT_DIR / "measurement_quality.csv"

quality_summary.to_csv(
    quality_path,
    index=False
)

# RESULTS
print("\nMISSION 1 RESULTS")
print("-----------------------------")

print("Selected Features:", selected_features)
print("Highest Missing Percentage:",
      f"{missing_percentages.max():.2f}%")

print("Feature with Highest Flagged Percentage:",
      quality_summary.iloc[0]["Feature"])

print("Total Flagged Observations:",
      flags.sum())

print("\nQUALITY SUMMARY")
print(quality_summary.round(4))

print("\nVectorized Results Verified:",
      np.array_equal(flags.sum(axis=0), flagged_counts))

print("Output Created:", quality_path.is_file())



MISSION 1 RESULTS
-----------------------------
Selected Features: ['Feature_107', 'Feature_202', 'Feature_474', 'Feature_496', 'Feature_588', 'Feature_279']
Highest Missing Percentage: 3.25%
Feature with Highest Flagged Percentage: Feature_496
Total Flagged Observations: 351

QUALITY SUMMARY
       Feature     Mean   Median  Sample_SD  Missing_Percentage  \
3  Feature_496  29.8659  22.0391    24.6216              3.2546   
5  Feature_279   0.0411   0.0372     0.0203              0.1276   
2  Feature_474  37.6370  32.8200    24.8229              0.4467   
4  Feature_588   0.0053   0.0046     0.0029              0.0638   
1  Feature_202  10.1705   8.4620    14.6229              0.4467   
0  Feature_107  -0.0018   0.0000     0.0875              0.3829   

   Flagged_Count  Flagged_Percentage  
3            143              9.4327  
5             79              5.0479  
2             42              2.6923  
4             39              2.4904  
1             24              1.5385  
0

In [4]:

# MISSION 3
# STATISTICAL ROBUSTNESS AND SENSITIVITY ANALYSIS

# SELECT FEATURES WITH AT LEAST 10 OBSERVATIONS
# SELECT FEATURES WITH AT LEAST 10 OBSERVATIONS

all_feature_names = [
    f"Feature_{i:03d}"
    for i in range(raw_measurements.shape[1])
]

pass_mask = labels == -1
fail_mask = labels == 1

eligible_features = []

for i in usable_indices:

    pass_count = np.sum(
        ~np.isnan(raw_measurements[pass_mask, i])
    )

    fail_count = np.sum(
        ~np.isnan(raw_measurements[fail_mask, i])
    )

    if pass_count >= 10 and fail_count >= 10:
        eligible_features.append(i)

assert len(eligible_features) >= 5

# REPRODUCIBLE FEATURE SELECTION
rng = np.random.default_rng(RANDOM_SEED)

chosen_indices = rng.choice(
    eligible_features,
    size=5,
    replace=False
)

chosen_features = [
    all_feature_names[i] for i in chosen_indices
]

# TWO PREPARATION STRATEGIES
available_data = raw_measurements[:, chosen_indices].copy()

selected_medians = np.nanmedian(
    available_data,
    axis=0
)

imputed_data = np.where(
    np.isnan(available_data),
    selected_medians,
    available_data
)

# BOOTSTRAP SETTINGS
BOOTSTRAP_REPETITIONS = 200 + (10 * SEED_DIGIT)

# FUNCTION FOR STRATIFIED BOOTSTRAP
def run_bootstrap(data, seed):

    rng = np.random.default_rng(seed)

    results = []

    for j, feature in enumerate(chosen_features):

        pass_values = data[
            pass_mask, j
        ]

        fail_values = data[
            fail_mask, j
        ]

        # AVAILABLE-CASE ANALYSIS
        pass_values = pass_values[
            ~np.isnan(pass_values)
        ]

        fail_values = fail_values[
            ~np.isnan(fail_values)
        ]

        differences = []

        # STRATIFIED BOOTSTRAP
        for _ in range(BOOTSTRAP_REPETITIONS):

            pass_sample = rng.choice(
                pass_values,
                size=len(pass_values),
                replace=True
            )

            fail_sample = rng.choice(
                fail_values,
                size=len(fail_values),
                replace=True
            )

            differences.append(
                fail_sample.mean() - pass_sample.mean()
            )

        differences = np.array(differences)

        mean_difference = (
            fail_values.mean() - pass_values.mean()
        )

        ci_lower, ci_upper = np.percentile(
            differences,
            [2.5, 97.5]
        )

        results.append({
            "Feature": feature,
            "Mean_Difference": mean_difference,
            "Bootstrap_Mean_Difference": differences.mean(),
            "CI_Lower": ci_lower,
            "CI_Upper": ci_upper
        })

    return pd.DataFrame(results)


# RUN BOTH PREPARATION STRATEGIES
available_results = run_bootstrap(
    available_data,
    RANDOM_SEED
)

imputed_results = run_bootstrap(
    imputed_data,
    RANDOM_SEED
)

available_results["Strategy"] = "Available Case"

imputed_results["Strategy"] = "Median Imputation"

# COMBINE RESULTS
robustness = pd.concat(
    [available_results, imputed_results],
    ignore_index=True
)

# COMPARE BOTH STRATEGIES
available_map = available_results.set_index("Feature")
imputed_map = imputed_results.set_index("Feature")

robustness["Difference_Change"] = (
    robustness["Feature"].map(
        (
            imputed_map["Mean_Difference"] -
            available_map["Mean_Difference"]
        ).abs()
    )
    if False else np.nan
)

# CALCULATE CHANGE IN MEAN DIFFERENCE
change_map = (
    imputed_map["Mean_Difference"] -
    available_map["Mean_Difference"]
).abs()

robustness["Difference_Change"] = (
    robustness["Feature"].map(change_map)
)

# CHECK DIFFERENCE SIGN
sign_changes = []

for feature in chosen_features:

    available_sign = np.sign(
        available_map.loc[feature, "Mean_Difference"]
    )

    imputed_sign = np.sign(
        imputed_map.loc[feature, "Mean_Difference"]
    )

    sign_changes.append(
        available_sign != imputed_sign
    )

# RANK FEATURES BY ABSOLUTE MEAN DIFFERENCE
available_ranks = (
    available_results.set_index("Feature")[
        "Mean_Difference"
    ].abs().rank(ascending=False, method="min")
)

imputed_ranks = (
    imputed_results.set_index("Feature")[
        "Mean_Difference"
    ].abs().rank(ascending=False, method="min")
)

ranking_changes = (
    available_ranks != imputed_ranks
)

# VERIFY REPRODUCIBILITY
repeat_available = run_bootstrap(
    available_data,
    RANDOM_SEED
)

repeat_imputed = run_bootstrap(
    imputed_data,
    RANDOM_SEED
)

reproducibility = (
    np.allclose(
        available_results[
            ["Mean_Difference", "Bootstrap_Mean_Difference",
             "CI_Lower", "CI_Upper"]
        ],
        repeat_available[
            ["Mean_Difference", "Bootstrap_Mean_Difference",
             "CI_Lower", "CI_Upper"]
        ]
    )
    and
    np.allclose(
        imputed_results[
            ["Mean_Difference", "Bootstrap_Mean_Difference",
             "CI_Lower", "CI_Upper"]
        ],
        repeat_imputed[
            ["Mean_Difference", "Bootstrap_Mean_Difference",
             "CI_Lower", "CI_Upper"]
        ]
    )
)

# CREATE INTERPRETATION REPORT
largest_change = change_map.max()

changed_sign_features = [
    chosen_features[i]
    for i in range(len(chosen_features))
    if sign_changes[i]
]

changed_ranking_features = list(
    ranking_changes[ranking_changes].index
)

# EXPORT CSV
robustness_path = OUTPUT_DIR / "robustness_comparison.csv"

robustness.to_csv(
    robustness_path,
    index=False
)

# GENERATE TEXT REPORT
report_path = OUTPUT_DIR / "sensitivity_report.txt"

report = f"""
STATISTICAL ROBUSTNESS AND SENSITIVITY ANALYSIS

Student: {LAST_NAME}
Student ID: {STUDENT_ID}

Selected Features: {chosen_features}

Bootstrap Repetitions: {BOOTSTRAP_REPETITIONS}

Largest Change in Mean Difference:
{largest_change:.4f}

Features with Changed Difference Sign:
{changed_sign_features}

Features with Changed Ranking:
{changed_ranking_features}

Reproducibility Verified: {reproducibility}

INTERPRETATION:

Available-case analysis excludes missing measurements
when calculating each feature's group statistics.

Median imputation replaces missing measurements using
the median of each selected feature from the full dataset.

The bootstrap confidence intervals describe the
uncertainty in the estimated Fail-minus-Pass mean
differences under each preparation strategy.

Differences between the two strategies indicate
sensitivity to missing-value treatment.
"""

report_path.write_text(
    report,
    encoding="utf-8"
)

# DISPLAY RESULTS
print("\nMISSION 3 RESULTS")
print("-----------------------------")

print("Selected Features:", chosen_features)

print("Bootstrap Repetitions:",
      BOOTSTRAP_REPETITIONS)

print("\nROBUSTNESS COMPARISON")
print(robustness.round(4))

print("\nLargest Change in Mean Difference:",
      f"{largest_change:.4f}")

print("Features with Changed Difference Sign:",
      changed_sign_features)

print("Features with Changed Ranking:",
      changed_ranking_features)

print("Reproducibility Verified:", reproducibility)

print("\nOUTPUT FILES")
print("Robustness CSV Created:", robustness_path.is_file())
print("Sensitivity Report Created:", report_path.is_file())


MISSION 3 RESULTS
-----------------------------
Selected Features: ['Feature_203', 'Feature_107', 'Feature_280', 'Feature_499', 'Feature_279']
Bootstrap Repetitions: 240

ROBUSTNESS COMPARISON
       Feature  Mean_Difference  Bootstrap_Mean_Difference  CI_Lower  \
0  Feature_203           3.1840                     3.2957   -0.6602   
1  Feature_107          -0.0053                    -0.0049   -0.0239   
2  Feature_280          -0.0013                    -0.0013   -0.0021   
3  Feature_499         -39.1733                   -37.8775 -101.6411   
4  Feature_279          -0.0031                    -0.0035   -0.0069   
5  Feature_203           3.1830                     3.0323   -1.1232   
6  Feature_107          -0.0054                    -0.0048   -0.0254   
7  Feature_280          -0.0013                    -0.0013   -0.0021   
8  Feature_499         -38.8099                   -38.0334 -105.4624   
9  Feature_279          -0.0031                    -0.0030   -0.0065   

   CI_Upper  